# 2.2 · Searching sorted data

*Sequences* · 16 exercises

Keep data in order and search it in O(log N): binary search, lower bounds and their variations.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-2.2) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-2.2) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-2.2)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 2.2.1 · Ordered array insert  `Core`

**Task.** Create `OrderedArray` holding a Python list `items` that is always sorted. Write `insert(value)`: scan from the left to find the first element larger than `value`, then shift everything from there one slot right and write the value. Do not use `list.insert` or `sort`.

**Example.** `[3, 17, 80, 202]` -> `insert(75)` -> `[3, 17, 75, 80, 202]`

**Big O.** O(N). Comparisons up to the spot plus shifts after it always total about N + 2 steps.

**Note.** Insertion is slower than in a plain array (which appends in O(1)), the price paid for O(log N) search.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-1)

In [ ]:
# 2.2.1 Ordered array insert
class OrderedArray:
    def __init__(self, values=()):
        raise NotImplementedError

    def insert(self, value):
        raise NotImplementedError

In [ ]:
# Test helpers: run once.
def check_binary_search(fn):
    assert fn([3, 17, 75, 80, 202], 80) == 3
    assert fn([1, 3, 5, 7, 9], 1) == 0 and fn([1, 3, 5, 7, 9], 9) == 4
    assert fn([1, 3, 5], 4) == -1 and fn([], 1) == -1

In [ ]:
def test_ex_2_2_1():
    o = OrderedArray([3, 17, 80, 202])
    o.insert(75)
    assert o.items == [3, 17, 75, 80, 202]
    o.insert(1)
    o.insert(500)
    assert o.items == [1, 3, 17, 75, 80, 202, 500]
    e = OrderedArray()
    e.insert(4)
    assert e.items == [4]


test_ex_2_2_1()
print('2.2.1 passed')

### 2.2.6 · Insert using binary search for the position  `Practice`

**Task.** Add `OrderedArray.insert_fast(value)` that finds the position with your `lower_bound` from 2.2.5 instead of a linear scan, then shifts.

**Example.** Same result as `insert`, with only about log2(N) comparisons

**Big O.** Still O(N) overall because the shifts dominate; the comparisons drop to O(log N).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-6)

In [ ]:
# 2.2.6 Insert using binary search for the position
def insert_fast(self, value):
    raise NotImplementedError


OrderedArray.insert_fast = insert_fast

### 2.2.8 · Delete from an ordered array  `Core`

**Task.** Add `OrderedArray.delete(value)`: find it with binary search, shift the rest left, return `True`; return `False` if absent.

**Example.** `[2, 4, 6, 8]` -> `delete(4)` -> `[2, 6, 8]`

**Big O.** O(log N) to find + O(N) to shift = O(N).

*Its test runs further down, once the code it relies on has been defined.*

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-8)

In [ ]:
# 2.2.8 Delete from an ordered array
def delete(self, value):
    raise NotImplementedError


OrderedArray.delete = delete

### 2.2.2 · Linear search with early exit  `Core`

**Task.** Write `linear_search_ordered(arr, target)` for a sorted list. Return the index, or `-1` as soon as you pass a value larger than the target.

**Example.** `[3, 17, 75, 80, 202]`, target `22` -> `-1` after inspecting only 3 elements

**Big O.** O(N) worst case, but it often stops early.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-2)

In [ ]:
# 2.2.2 Linear search with early exit
def linear_search_ordered(arr, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_2():
    assert linear_search_ordered([3, 17, 75, 80, 202], 22) == -1
    assert linear_search_ordered([3, 17, 75, 80, 202], 75) == 2
    assert linear_search_ordered([], 1) == -1


test_ex_2_2_2()
print('2.2.2 passed')

### 2.2.3 · Binary search, iterative  `Core`

**Task.** Write `binary_search(arr, target)` on a sorted list using `lower` and `upper` bounds and a midpoint. Return the index or `-1`.

**Example.** `[3, 17, 75, 80, 202]`, target `80` -> `3`

**Big O.** O(log N). Each step halves the range: 100 elements need at most 7 steps, 1,000,000 need 20.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-3)

In [ ]:
# 2.2.3 Binary search, iterative
def binary_search(arr, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_3():
    check_binary_search(binary_search)


test_ex_2_2_3()
print('2.2.3 passed')

In [ ]:
def test_ex_2_2_8():
    d = OrderedArray([2, 4, 6, 8])
    assert d.delete(4)
    assert d.items == [2, 6, 8]
    assert not d.delete(5)
    assert d.delete(8)
    assert d.items == [2, 6]


test_ex_2_2_8()
print('2.2.8 passed')

### 2.2.4 · Binary search, recursive  `Practice`

**Task.** Write `binary_search_recursive(arr, target, lower=0, upper=None)` with the same contract, using recursion instead of a loop.

**Example.** `[1, 3, 5, 7, 9]`, target `1` -> `0`

**Big O.** O(log N) time, O(log N) call-stack space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-4)

In [ ]:
# 2.2.4 Binary search, recursive
def binary_search_recursive(arr, target, lower=0, upper=None):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_4():
    check_binary_search(binary_search_recursive)


test_ex_2_2_4()
print('2.2.4 passed')

### 2.2.5 · Insertion point (lower bound)  `Core`

**Task.** Write `lower_bound(arr, target)`: the first index whose value is `>= target` (or `len(arr)` if none). You will reuse it in 2.6.

**Example.** `[10, 20, 20, 30]`, target `20` -> `1`; target `25` -> `3`; target `99` -> `4`

**Big O.** O(log N).

**Note.** The range is half-open `[lo, hi)`, so `hi` starts at `len(arr)`, not `len(arr) - 1`.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-5)

In [ ]:
# 2.2.5 Insertion point (lower bound)
def lower_bound(arr, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_5():
    assert lower_bound([10, 20, 20, 30], 20) == 1
    assert lower_bound([10, 20, 20, 30], 25) == 3
    assert lower_bound([10, 20, 20, 30], 99) == 4
    assert lower_bound([10, 20], 5) == 0
    assert lower_bound([], 5) == 0


test_ex_2_2_5()
print('2.2.5 passed')

In [ ]:
def test_ex_2_2_6():
    f = OrderedArray()
    for v in [5, 1, 4, 1, 9]:
        f.insert_fast(v)
    assert f.items == [1, 1, 4, 5, 9]
    g = OrderedArray([3, 17, 80, 202])
    g.insert_fast(75)
    assert g.items == [3, 17, 75, 80, 202]


test_ex_2_2_6()
print('2.2.6 passed')

### 2.2.7 · First and last occurrence  `Practice`

**Task.** Write `first_and_last(arr, target)` returning `(first_index, last_index)` in a sorted list with duplicates, or `(-1, -1)`. Use two binary searches.

**Example.** `[1, 2, 2, 2, 5]`, target `2` -> `(1, 3)`

**Big O.** O(log N). A linear scan outward from one match would degrade to O(N) when everything matches.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-7)

In [ ]:
# 2.2.7 First and last occurrence
def first_and_last(arr, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_7():
    assert first_and_last([1, 2, 2, 2, 5], 2) == (1, 3)
    assert first_and_last([1, 2, 5], 3) == (-1, -1)
    assert first_and_last([2.5, 2.5], 2.5) == (0, 1)


test_ex_2_2_7()
print('2.2.7 passed')

### 2.2.9 · Count the steps: linear vs binary  `Practice`

**Task.** Write `search_steps(n)` that searches `list(range(n))` for a missing value (`n`) both ways and returns `(linear_steps, binary_steps)`.

**Example.** `search_steps(100)` -> `(100, 7)`; `search_steps(10_000)` -> `(10000, 14)`

**Big O.** Linear grows 1:1 with N; binary grows by one step each time N doubles.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-9)

In [ ]:
# 2.2.9 Count the steps: linear vs binary
def search_steps(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_9():
    assert search_steps(100) == (100, 7)
    assert search_steps(10000) == (10000, 14)


test_ex_2_2_9()
print('2.2.9 passed')

### 2.2.10 · Integer square root by binary search  `Stretch`

**Task.** Write `int_sqrt(n)` returning the largest integer `r` with `r * r <= n`, without `math.sqrt`. Binary-search the answer between 0 and n.

**Example.** `int_sqrt(27)` -> `5`; `int_sqrt(36)` -> `6`

**Big O.** O(log N). Binary search works on any sorted range of candidate answers, not only on arrays.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-10)

In [ ]:
# 2.2.10 Integer square root by binary search
def int_sqrt(n):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_10():
    assert int_sqrt(27) == 5
    assert int_sqrt(36) == 6
    assert int_sqrt(0) == 0
    assert int_sqrt(1) == 1


test_ex_2_2_10()
print('2.2.10 passed')

### 2.2.11 · Search a rotated sorted array  `Practice`

**Task.** `nums` held distinct values in ascending order and was then rotated (`[15, 18, 22, 3, 7, 9]`). Write `search_rotated(nums, target)` that returns the index of `target` or `-1`, in O(log N): at each step decide which half is sorted, then whether `target` lies inside it.

**Example.** `[15, 18, 22, 3, 7, 9], 7` -> `4`; `[15, 18, 22, 3, 7, 9], 10` -> `-1`

**Big O.** O(log N) time, O(1) space: at every midpoint one half is in order, and a range check on that half decides which way to go.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-11)

In [ ]:
# 2.2.11 Search a rotated sorted array
def search_rotated(nums, target):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_11():
    nums = [15, 18, 22, 3, 7, 9]
    assert [search_rotated(nums, x) for x in nums] == list(range(6))
    assert search_rotated(nums, 10) == -1
    assert search_rotated(nums, 1) == -1
    assert search_rotated([], 4) == -1
    assert search_rotated([5], 5) == 0
    assert search_rotated([1, 2, 3], 3) == 2
    assert search_rotated([3, 1], 1) == 1


test_ex_2_2_11()
print('2.2.11 passed')

### 2.2.12 · Find a peak  `Core`

**Task.** `nums` is non-empty and no two neighbours are equal. Write `find_peak(nums)` that returns the index of a value larger than both neighbours (past the ends counts as smaller), in O(log N): compare `nums[mid]` with `nums[mid + 1]` and keep the rising side. Any peak is accepted.

**Example.** `[1, 3, 2, 5, 4]` -> `3` (`1` would also be accepted)

**Big O.** O(log N) time, O(1) space: if the next value is larger, a peak must lie to the right, because the values cannot rise forever.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-12)

In [ ]:
# 2.2.12 Find a peak
def find_peak(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_12():

    def is_peak(nums, i):
        return (i == 0 or nums[i - 1] < nums[i]) and (i == len(nums) - 1 or nums[i + 1] < nums[i])
    assert find_peak([1, 3, 2, 5, 4]) == 3
    for nums in ([7], [1, 2, 3, 4], [4, 3, 2, 1], [2, 9, 1, 8, 0, 7, 6], [5, 1]):
        assert is_peak(nums, find_peak(nums))


test_ex_2_2_12()
print('2.2.12 passed')

### 2.2.13 · Binary search on the answer  `Practice`

**Task.** Write `smallest_max_load(loads, k)` that splits the positive `loads`, in order, into at most `k` contiguous groups so the largest group sum is as small as possible, and returns that sum. Binary search the answer between `max(loads)` and `sum(loads)`; to test a limit, fill groups greedily and count them.

**Example.** `[4, 9, 3, 6, 2, 8], 3` -> `13` (`[4, 9] [3, 6, 2] [8]`)

**Big O.** O(N log S) time for loads summing to S, O(1) space: whether a limit works is monotonic, so binary search finds the smallest one, and each check is a greedy pass.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-13)

In [ ]:
# 2.2.13 Binary search on the answer
def smallest_max_load(loads, k):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_13():
    loads = [4, 9, 3, 6, 2, 8]
    assert smallest_max_load(loads, 3) == 13
    assert smallest_max_load(loads, 1) == 32
    assert smallest_max_load(loads, 6) == 9
    assert smallest_max_load([5, 5, 5, 5], 2) == 10
    assert smallest_max_load([7], 3) == 7


test_ex_2_2_13()
print('2.2.13 passed')

### 2.2.14 · Index equal to value  `Core`

**Task.** `nums` holds distinct integers in ascending order. Write `fixed_point(nums)` that returns the smallest index `i` with `nums[i] == i`, or `-1`, in O(log N).

**Example.** `[-4, -1, 2, 5, 9]` -> `2`; `[1, 2, 3]` -> `-1`

**Big O.** O(log N) time, O(1) space: with distinct sorted integers, `nums[i] - i` never decreases, so the matches form one block that binary search can find.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-14)

In [ ]:
# 2.2.14 Index equal to value
def fixed_point(nums):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_14():
    assert fixed_point([-4, -1, 2, 5, 9]) == 2
    assert fixed_point([1, 2, 3]) == -1
    assert fixed_point([0, 1, 2]) == 0
    assert fixed_point([-3, -1, 1, 3, 4]) == 3
    assert fixed_point([]) == -1
    assert fixed_point([-5, 0, 7]) == -1


test_ex_2_2_14()
print('2.2.14 passed')

### 2.2.15 · Median of two sorted arrays  `Stretch`

**Task.** `a` and `b` are sorted and not both empty. Write `median_of_two(a, b)` that returns the median of all their values as a float, in O(log min(M, N)) without merging: binary search how many values of the shorter array go in the lower half, and check the four values around the two cuts.

**Example.** `[1, 4, 9], [2, 3]` -> `3.0`; `[1, 4], [2, 3]` -> `2.5`

**Big O.** O(log min(M, N)) time, O(1) space: binary search the cut in the shorter array; the cut in the other follows, and four border values say which way to move.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-15)

In [ ]:
# 2.2.15 Median of two sorted arrays
def median_of_two(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_2_2_15():
    assert median_of_two([1, 4, 9], [2, 3]) == 3.0
    assert median_of_two([1, 4], [2, 3]) == 2.5
    assert median_of_two([], [5]) == 5.0
    assert median_of_two([1, 2], [3, 4, 5, 6]) == 3.5
    assert median_of_two([7], [1, 2, 3]) == 2.5
    assert median_of_two([10, 20, 30], []) == 20.0
    assert median_of_two([1, 1], [1, 1]) == 1.0


test_ex_2_2_15()
print('2.2.15 passed')

### 2.2.16 · Weighted random pick  `Practice`

**Task.** Build `WeightedPicker(weights)` for positive integer weights with `total()` and `pick(r)`: for an integer `r` in `[0, total())`, `pick` returns the index whose range of prefix sums holds `r`, found by binary search. Then `pick(random.randrange(picker.total()))` returns index `i` with probability `weights[i] / total`.

**Example.** `WeightedPicker([3, 1, 2])`: `pick(0)` -> `0`; `pick(3)` -> `1`; `pick(5)` -> `2`

**Big O.** O(N) to build, O(log N) per pick, O(N) space: the prefix sums split `[0, total)` into one range per index, and binary search finds the range holding `r`.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-2-2-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-2-2-16)

In [ ]:
# 2.2.16 Weighted random pick
class WeightedPicker:
    def __init__(self, weights):
        raise NotImplementedError

    def total(self):
        raise NotImplementedError

    def pick(self, r):
        raise NotImplementedError

In [ ]:
def test_ex_2_2_16():
    picker = WeightedPicker([3, 1, 2])
    assert picker.total() == 6
    assert [picker.pick(r) for r in range(6)] == [0, 0, 0, 1, 2, 2]
    weights = [5, 1, 4, 2]
    picker = WeightedPicker(weights)
    counts = [0] * len(weights)
    for r in range(picker.total()):
        counts[picker.pick(r)] += 1
    assert counts == weights
    assert WeightedPicker([9]).pick(8) == 0


test_ex_2_2_16()
print('2.2.16 passed')